# QueryGenie — Week 4 Gate: Getting CodeS Running

**Project:** QueryGenie: A Self-Correcting Natural Language to SQL Interface  
**Team 21** — 2320090050 (N V V S S Jayakanth Kamisetti), 2320090009 (Bodupally Narendar)  
**Supervisor:** M Parameswar

**Anchor paper:** CodeS — *Towards Building Open-source Language Models for Text-to-SQL* (SIGMOD 2024)  
arXiv: https://arxiv.org/abs/2402.16347 · Code: https://github.com/RUCKBReasoning/codes

---

## How to use this notebook

It is deliberately split into two tiers.

| Tier | What it does | Time | Risk | Satisfies |
|---|---|---|---|---|
| **1 — Smoke test** | Loads `seeklhy/codes-1b` from Hugging Face and generates SQL from a schema + question | ~10 min | Low | SOP Week-4 gate: *"run the original code once to verify it works"* |
| **2 — Full reproduction** | Recreates the authors' exact environment and runs their evaluation scripts on Spider | Hours | Medium–High | Month 2 reproduction (45% of the grade) |

**Run Tier 1 first.** It clears the Week-4 rejection gate quickly. Only then attempt Tier 2.

> **Runtime → Change runtime type → T4 GPU** before running anything.


---
## 0 · Environment check

Records the hardware you are running on. These values go into the reproduction log,
since the SOP requires hardware to be documented when explaining any metric deviation.


In [ ]:
import sys, platform, subprocess, json, datetime

info = {
    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
}

try:
    import torch
    info['torch'] = torch.__version__
    info['cuda_available'] = torch.cuda.is_available()
    if torch.cuda.is_available():
        info['gpu'] = torch.cuda.get_device_name(0)
        info['gpu_memory_GB'] = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
except ImportError:
    info['torch'] = 'not installed'

for k, v in info.items():
    print(f'{k:18}: {v}')

if not info.get('cuda_available'):
    print('\nWARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU')


---
# TIER 1 — Smoke test

Goal: prove the released CodeS model loads and produces SQL. This is the evidence for the
Week-4 gate. It uses a current `transformers` rather than the repo's pinned 4.28.1, so it is
**not** a faithful reproduction — it is a liveness check. Tier 2 does the faithful run.


In [ ]:
!pip -q install --upgrade transformers accelerate sentencepiece
print('done')


In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'seeklhy/codes-1b'   # 3b / 7b / 15b also available

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
model.eval()
print(f'loaded {MODEL} in {time.time()-t0:.1f}s')
print('parameters:', f'{sum(p.numel() for p in model.parameters())/1e9:.2f}B')


### Prompt format

CodeS is trained to continue a prompt that states the database schema followed by the question.
The repo builds this prompt programmatically in `prepare_sft_datasets.py` (with schema filtering
and BM25-matched cell values). Below is a simplified version for the smoke test — expect the
faithful format in Tier 2 to differ, which is itself worth noting in your reproducibility report.


In [ ]:
def build_prompt(schema: str, question: str) -> str:
    return f'{schema}\n{question}\n'

schema = (
    'database schema :\n'
    'table student , columns = [ student.student_id ( int | primary key ) , '
    'student.name ( text ) , student.department ( text ) , student.year ( int ) ]\n'
    'table result , columns = [ result.result_id ( int | primary key ) , '
    'result.student_id ( int ) , result.subject ( text ) , result.marks ( int ) ]\n'
    'foreign keys : result.student_id = student.student_id'
)

question = 'Which students failed more than two subjects? A subject is failed if marks are below 40.'

prompt = build_prompt(schema, question)
print(prompt)


In [ ]:
import time

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

t0 = time.time()
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
    )
elapsed = time.time() - t0

generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- generated SQL ---')
print(generated.strip())
print(f'\ninference time: {elapsed:.2f}s')


### Sanity-check the SQL actually executes

Builds a throwaway SQLite database matching the schema above and runs whatever the model
produced. This is the seed of your **execution-guided self-correction** enhancement: the same
try/except is what will later trigger a repair attempt.


In [ ]:
import sqlite3, re

con = sqlite3.connect(':memory:')
cur = con.cursor()
cur.executescript('''
CREATE TABLE student (student_id INTEGER PRIMARY KEY, name TEXT, department TEXT, year INTEGER);
CREATE TABLE result (result_id INTEGER PRIMARY KEY, student_id INTEGER, subject TEXT, marks INTEGER);
INSERT INTO student VALUES (1,'Asha','CSE',3),(2,'Ravi','CSE',3),(3,'Meera','IT',2);
INSERT INTO result VALUES
  (1,1,'DBMS',35),(2,1,'OS',30),(3,1,'CN',72),
  (4,2,'DBMS',55),(5,2,'OS',61),(6,2,'CN',48),
  (7,3,'DBMS',20),(8,3,'OS',25),(9,3,'CN',38);
''')
con.commit()

sql = generated.strip().split(';')[0].strip()
print('executing:', sql, '\n')

try:
    rows = cur.execute(sql).fetchall()
    print('EXECUTED OK ->', rows)
    exec_ok = True
except Exception as e:
    print('EXECUTION FAILED ->', type(e).__name__, e)
    exec_ok = False

print('\nexpected answer: Asha (2 fails) and Meera (3 fails) -> students with >2 failed subjects: Meera')


### Record the Tier-1 run

Appends a row to `reproduction_log.csv`. The SOP requires the original code to be run **three
times** with metrics, runtime, seeds and hardware recorded (Week 5, 20% of the grade), so start
logging from the very first run.


In [ ]:
import csv, os, datetime

LOG = 'reproduction_log.csv'
FIELDS = ['run_id','timestamp','tier','model','dataset','seed','metric_name',
          'metric_value','runtime_s','gpu','torch_version','transformers_version','notes']

import transformers
row = {
    'run_id': 'T1-001',
    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
    'tier': 'Tier 1 - smoke test',
    'model': MODEL,
    'dataset': 'hand-built toy schema (not Spider)',
    'seed': 'n/a (beam search, deterministic)',
    'metric_name': 'query_executed_without_error',
    'metric_value': str(exec_ok),
    'runtime_s': round(elapsed, 2),
    'gpu': info.get('gpu', 'cpu'),
    'torch_version': info.get('torch'),
    'transformers_version': transformers.__version__,
    'notes': 'Liveness check only. Not a faithful reproduction - modern transformers, simplified prompt.',
}

new = not os.path.exists(LOG)
with open(LOG, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if new:
        w.writeheader()
    w.writerow(row)

print('logged to', LOG)
print(open(LOG).read())


> **Week-4 gate cleared** once the cells above run without error. Download `reproduction_log.csv`
> and screenshot the generated SQL — that is your evidence. Commit both to the GitHub repo.


---
# TIER 2 — Faithful reproduction

This follows the authors' documented procedure. Read the warnings before starting.

**Known friction points:**

1. The repo pins Python **3.8.5**, PyTorch **1.13.1**, transformers **4.28.1**, `scipy==1.5.4`.
   Modern Colab runs Python 3.11+, where some pins will not build. We use `condacolab` to create
   a genuine 3.8 environment — note this **restarts the kernel once**, which is expected.
2. `pyserini` needs **Java 11**.
3. `SimCSE` installs from a custom fork, not PyPI.
4. `data.zip`, `sic_ckpts.zip` and `test_suite_sql_eval.zip` are hosted on **Google Drive**.
   Large Drive downloads are rate-limited; if `gdown` fails, download manually and upload.
5. The authors evaluated on 8× A800 80GB. A single T4 can run CodeS-1B inference, but expect
   long runtimes and possible OOM on larger scales.

**If Tier 2 proves unworkable**, this is exactly the SOP's Week-4 decision point. Fallback:
**RESDSQL** (https://github.com/RUCKBReasoning/RESDSQL) — same research group, same benchmark,
released checkpoints — so switching stays inside the approved topic. Document the reason.


### 2.1 · Install Java 11


In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-11-jdk > /dev/null
!java -version


### 2.2 · Clone the official repository


In [ ]:
%cd /content
!git clone https://github.com/RUCKBReasoning/codes.git
%cd /content/codes
!ls


### 2.3 · Create the Python 3.8.5 environment

`condacolab.install()` **restarts the kernel**. That is normal — after the restart, skip
straight to the next cell (do not re-run this one).


In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()   # kernel restarts here


In [ ]:
# Run AFTER the kernel restart
!conda create -n codes python=3.8.5 -y
!conda run -n codes pip install torch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 --index-url https://download.pytorch.org/whl/cu117
%cd /content/codes
!conda run -n codes pip install -r requirements.txt


### 2.4 · Install the SimCSE fork


In [ ]:
%cd /content/codes
!git clone https://github.com/lihaoyang-ruc/SimCSE.git
%cd SimCSE
!conda run -n codes python setup.py install
%cd /content/codes


### 2.5 · Download datasets, checkpoints and evaluation scripts

Google Drive IDs taken from the official README. If a download stalls or returns an HTML error
page, open the link in a browser, download manually, and upload to `/content/codes/`.


In [ ]:
!pip -q install gdown
%cd /content/codes

FILES = {
    'data.zip':                 '189spLXUL3gF8k4sny5qiWMqW3wOzx5AD',
    'sic_ckpts.zip':            '1V3F4ihTSPbV18g3lrg94VMH-kbWR_-lY',
    'test_suite_sql_eval.zip':  '1iNa1WgA9tN_OFna08nq_tHZdXx9Lz2vO',
}

import subprocess, os
for name, fid in FILES.items():
    if os.path.exists(name):
        print('already have', name); continue
    print('downloading', name)
    subprocess.run(['gdown', '--id', fid, '-O', name])

!ls -lh *.zip


In [ ]:
%cd /content/codes
!unzip -q -o data.zip
!unzip -q -o sic_ckpts.zip
!unzip -q -o test_suite_sql_eval.zip
!ls


### 2.6 · Inspect the evaluation scripts before running

Read these before executing. They are written for a multi-GPU machine — you will likely need to
reduce the model scale to `codes-1b` and cut the batch size. **Record every change you make**,
because deviations must be documented in the reproducibility report.


In [ ]:
!echo '===== run_sft_evaluations.sh ====='; cat run_sft_evaluations.sh
!echo; echo '===== run_few_shot_evaluations.sh ====='; cat run_few_shot_evaluations.sh


### 2.7 · Run the evaluation

Edit the command below to match the script's actual arguments and your chosen scale.
Start with the **smallest** configuration (`codes-1b`, Spider) to get a result at all, then scale up.


In [ ]:
%cd /content/codes
# Inspect the script first (cell 2.6), then run the specific python command it wraps, e.g.:
# !conda run -n codes python -u text2sql.py --llm_path seeklhy/codes-1b --dataset_name spider ...
#
# Uncomment to run the whole script as-is:
# !conda run -n codes bash run_sft_evaluations.sh
print('Read run_sft_evaluations.sh above, then run the appropriate command here.')


### 2.8 · Log the Tier-2 run

Fill in the metric the evaluation prints (Execution Accuracy / Exact Match) and run this three
times to satisfy the Week-5 requirement.


In [ ]:
import csv, os, datetime

# ---- edit these after each evaluation run ----
RUN_ID       = 'T2-001'
SEED         = 42
METRIC_NAME  = 'execution_accuracy_spider_dev'
METRIC_VALUE = None      # e.g. 0.771
RUNTIME_S    = None      # wall-clock seconds
NOTES        = 'CodeS-1B, single T4, batch size reduced from repo default.'
# ----------------------------------------------

assert METRIC_VALUE is not None, 'Fill in METRIC_VALUE from the evaluation output first.'

LOG = 'reproduction_log.csv'
FIELDS = ['run_id','timestamp','tier','model','dataset','seed','metric_name',
          'metric_value','runtime_s','gpu','torch_version','transformers_version','notes']

row = {
    'run_id': RUN_ID,
    'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
    'tier': 'Tier 2 - faithful reproduction',
    'model': 'seeklhy/codes-1b',
    'dataset': 'Spider dev',
    'seed': SEED,
    'metric_name': METRIC_NAME,
    'metric_value': METRIC_VALUE,
    'runtime_s': RUNTIME_S,
    'gpu': info.get('gpu', 'unknown'),
    'torch_version': '1.13.1',
    'transformers_version': '4.28.1',
    'notes': NOTES,
}

new = not os.path.exists(LOG)
with open(LOG, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if new: w.writeheader()
    w.writerow(row)
print(open(LOG).read())


### 2.9 · Compare against the paper

The SOP defines reproduction as complete when your numbers land within **±5–10%** of the
published values, *or* the deviation is documented with a stated cause. Read the reported figure
for your exact configuration out of the CodeS paper (do not rely on memory) and enter it below.


In [ ]:
PAPER_REPORTED = None   # <- read this from the CodeS paper for YOUR exact config
OURS           = None   # <- from your run above

if PAPER_REPORTED and OURS:
    diff = OURS - PAPER_REPORTED
    pct  = diff / PAPER_REPORTED * 100
    print(f'paper reported : {PAPER_REPORTED:.4f}')
    print(f'ours           : {OURS:.4f}')
    print(f'difference     : {diff:+.4f}  ({pct:+.2f}%)')
    print()
    if abs(pct) <= 10:
        print('WITHIN TOLERANCE -> reproduction complete.')
    else:
        print('OUTSIDE TOLERANCE -> you must document the cause:')
        print('  - different GPU / precision (fp16 vs bf16)')
        print('  - different library versions')
        print('  - reduced batch size or beam width')
        print('  - different model scale than the reported row')
else:
    print('Fill in both values.')


---
## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `gdown` returns an HTML file | Drive rate limit / quota | Download in a browser, upload manually |
| `scipy==1.5.4` fails to build | Running Python 3.11+, not 3.8 | Use the conda env (2.3); confirm with `conda run -n codes python -V` |
| `JavaNotFoundError` from pyserini | Java missing | Re-run 2.1; check `java -version` |
| CUDA out of memory | Model too large for the GPU | Use `codes-1b`, reduce batch size, or use fp16 |
| Colab disconnects mid-run | Session timeout | Save checkpoints to Drive; run overnight in smaller chunks |
| Results differ slightly from the paper | transformers version drift (the authors warn about this) | Document it — this is a legitimate, expected deviation |

---

## Next steps once Tier 2 produces a number

1. Run the evaluation **three times** and complete `reproduction_log.csv` (Week 5 deliverable).
2. **Ablation:** disable the schema filter and re-measure — the drop is your Week-7 deliverable.
3. Write **Reproducibility Report v1.0**: what matched, what did not, and why (Week 8).
4. Begin the enhancement: execution-guided self-correction, then confidence-aware abstention.
